# 15. Case 2's CAV-RAG loop: modality mixing, concept vectors, text-conditioned attribution, and a deterministic-verdict fix

Extends the RAG-CAV loop (`14_rag_cav_loop.ipynb`) from Case 1 to Case 2, using the shared brain-text embedding space's own structure rather than porting Case 1's machinery unchanged. Four parts:

1. **Modality-gap diagnostics** — retrieval precision alone can't tell whether brain and text embeddings are genuinely mixed or just well-ranked; three direct checks.
2. **Case2-native CAV** — concept directions derived purely from text-prototype differences, no labeled brain examples needed.
3. **Text-conditioned attribution** — a capability Case 1 cannot offer: RSN attribution for *arbitrary* free text, not just the 6 known conditions.
4. **The CAV-RAG loop, with a deterministic-verdict fix** — the loop's original design let the LLM freely judge agreement between literature and CAV evidence; measured, it defaulted to AGREE regardless of the actual TCAV score. Fixed here by computing the verdict from (stance, TCAV) in code and asking the LLM only to narrate it, not decide it.


In [1]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import torch


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "environment.yml").exists() or (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate the NeuroLens repository root.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from neurolens.data_setup import make_dataloaders
from neurolens.model_builder import TransformerDecoder
from neurolens.contrastive import (
    ContrastiveModel, TextPrototypeEncoder, encode_condition_prototypes, extract_brain_embeddings,
    get_device, run_epoch, text_to_brain_retrieval_metrics,
)
from neurolens.concepts_case2 import run_case2_concept_analysis
from neurolens.interpretability import load_roi_to_network, network_roi_indices, NETWORK_NAMES
from neurolens.interpretability_case2 import compare_methods_text
from neurolens.retrieval import load_index, load_embedding_model, load_reranker
from neurolens.pipeline import explain_decoded_window_with_cav_loop_case2, make_mlx_generate_fn

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "hcp_ya_s1200" / "runs"
RESULTS_DIR = PROJECT_ROOT / "results"
EMBED_DIM = 64
BACKBONE_DIM = 128
CLASS_TO_CONDITION = {
    "0": "baseline", "1": "left_hand", "2": "right_hand",
    "3": "left_foot", "4": "right_foot", "5": "tongue",
}
CLASS_NAMES = [CLASS_TO_CONDITION[str(i)] for i in range(6)]

device = get_device()
print("device:", device)

train_loader, val_loader, test_loader, info = make_dataloaders(PROCESSED_ROOT, batch_size=64, window_length=32)
print("n_test_windows =", len(test_loader.dataset))

embedding_model_st = load_embedding_model(device="cpu")
text_embeddings = encode_condition_prototypes(embedding_model_st, CLASS_TO_CONDITION)
text_encoder = TextPrototypeEncoder(text_embeddings, embed_dim=EMBED_DIM)
backbone = TransformerDecoder(include_hrf_head=False)
model = ContrastiveModel(backbone, backbone_dim=BACKBONE_DIM, text_encoder=text_encoder, embed_dim=EMBED_DIM)
model.load_state_dict(torch.load(PROJECT_ROOT / "models" / "case2_transformer_w32" / "best.pt", map_location="cpu"))
model.to(device)
model.eval()

test_metrics = run_epoch(model, test_loader, device, optimizer=None)
print(f"sanity check - test macro-F1: {test_metrics['macro_f1']:.4f}")

roi_labels_path = PROCESSED_ROOT / "sub-100307" / "tfMRI_MOTOR_LR" / "roi_labels.tsv"
network_indices = network_roi_indices(load_roi_to_network(roi_labels_path))


/Users/srinivasgovindasurampudi/miniconda3/envs/neurolens/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


n_test_windows = 3048


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6853.44it/s]

sanity check - test macro-F1: 0.8826


## Part 1 — Are the brain and text embeddings genuinely mixed?

Text-to-brain retrieval precision alone can be perfect while recall is tiny (a small `k` against a large true-class pool), and even perfect *ranking* is compatible with a "modality gap" (Liang et al., 2022) — two well-separated clouds that still rank correctly relative to each other. Three checks: precision/recall/F1 at k (not precision alone), a centroid-gap ratio, a brain-only silhouette score, and a 2D visualization.

In [2]:
brain_z, brain_labels = extract_brain_embeddings(model, test_loader, device)
with torch.no_grad():
    text_z = model.text_encoder().cpu().numpy()
print("brain_z:", brain_z.shape, "text_z:", text_z.shape)

metrics = text_to_brain_retrieval_metrics(brain_z, brain_labels, text_z, top_k_values=[5, 10, 20, 50, 100])
for c in range(6):
    n_rel = metrics[c][5]["n_relevant"]
    r_precision_k = n_rel
    print(f"{CLASS_NAMES[c]}: n_relevant={n_rel}  precision@5={metrics[c][5]['precision']:.2f}  "
          f"recall@5={metrics[c][5]['recall']:.3f}  recall@100={metrics[c][100]['recall']:.3f}")


brain_z: (3048, 64) text_z: (6, 64)
baseline: n_relevant=576  precision@5=1.00  recall@5=0.009  recall@100=0.174
left_hand: n_relevant=468  precision@5=1.00  recall@5=0.011  recall@100=0.209
right_hand: n_relevant=468  precision@5=0.80  recall@5=0.009  recall@100=0.203
left_foot: n_relevant=504  precision@5=1.00  recall@5=0.010  recall@100=0.196
right_foot: n_relevant=504  precision@5=1.00  recall@5=0.010  recall@100=0.194
tongue: n_relevant=528  precision@5=1.00  recall@5=0.009  recall@100=0.188


In [3]:
r_precision = {}
for c in range(6):
    n_relevant = int((brain_labels == c).sum())
    scores = brain_z @ text_z[c]
    ranked = np.argsort(scores)[::-1]
    top_r = ranked[:n_relevant]
    r_precision[CLASS_NAMES[c]] = float((brain_labels[top_r] == c).mean())
print("R-precision (= recall = F1 at k = class size) per class:")
for k, v in r_precision.items():
    print(f"  {k}: {v:.3f}")
print(f"mean: {np.mean(list(r_precision.values())):.3f}")


R-precision (= recall = F1 at k = class size) per class:
  baseline: 0.889
  left_hand: 0.844
  right_hand: 0.891
  left_foot: 0.863
  right_foot: 0.863
  tongue: 0.828
mean: 0.863


In [4]:
rng = np.random.default_rng(0)
sample_idx = rng.choice(len(brain_z), size=min(2000, len(brain_z)), replace=False)
brain_sample = brain_z[sample_idx]

brain_centroid = brain_sample.mean(axis=0)
text_centroid = text_z.mean(axis=0)

within_brain_pairs = rng.choice(len(brain_sample), size=(3000, 2))
mean_within_brain_dist = float(np.linalg.norm(
    brain_sample[within_brain_pairs[:, 0]] - brain_sample[within_brain_pairs[:, 1]], axis=1
).mean())

cross_dists = [np.linalg.norm(brain_z[i] - text_z[j]) for i in sample_idx[:2000] for j in range(6)]
mean_cross_modal_dist = float(np.mean(cross_dists))
gap_ratio = mean_cross_modal_dist / mean_within_brain_dist

from sklearn.metrics import silhouette_score
brain_only_silhouette = float(silhouette_score(brain_sample, brain_labels[sample_idx], metric="cosine"))

print(f"cross-modal / within-brain distance ratio: {gap_ratio:.3f}  (near 1.0 = no systematic modality gap)")
print(f"brain-only silhouette score (text ignored): {brain_only_silhouette:.3f}")


cross-modal / within-brain distance ratio: 1.012  (near 1.0 = no systematic modality gap)
brain-only silhouette score (text ignored): 0.429


In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

combined = np.concatenate([brain_sample, text_z], axis=0)
pca = PCA(n_components=2, random_state=0)
combined_2d = pca.fit_transform(combined)
brain_2d, text_2d = combined_2d[: len(brain_sample)], combined_2d[len(brain_sample):]

colors = plt.cm.tab10(np.linspace(0, 1, 6))
fig, ax = plt.subplots(figsize=(7, 6))
for c in range(6):
    mask = brain_labels[sample_idx] == c
    ax.scatter(brain_2d[mask, 0], brain_2d[mask, 1], s=8, alpha=0.35, color=colors[c], label=f"brain: {CLASS_NAMES[c]}")
for c in range(6):
    ax.scatter(text_2d[c, 0], text_2d[c, 1], s=260, marker="*", color=colors[c], edgecolor="black",
               linewidth=1.2, label=f"text: {CLASS_NAMES[c]}", zorder=5)
ax.set_title("Case 2 joint embedding space (PCA): brain (dots) vs text (stars)")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
ax.legend(fontsize=6, ncol=2, loc="best")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "case2_embedding_space_pca.png", dpi=150)
plt.show()
print("saved figure")


saved figure


/var/folders/r4/spznqqj55yjg7yv8f_ym1pmm0000gn/T/ipykernel_73696/3912851519.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Reading the diagnostics**: the cross-modal/within-brain distance ratio near 1.0 means brain and text points are about as far from each other as brain points are from each other — no systematic offset. The brain-only silhouette score (computed with text entirely ignored) shows the brain encoder organizes windows by class on its own, not merely relative to the six fixed text anchors. And the figure should show each text-prototype star sitting inside its matching-color brain cluster, not off in a separate region. Together: the space is genuinely mixed, not just well-ranked for retrieval — R-precision (the single honest retrieval number, since raw precision@k is inflated by small k against a large true-class pool) sits at 0.83–0.89 across classes, consistent with the ~90% classification macro-F1.

## Part 2 — Case2-native CAV: concept directions from text alone

Case 1's CAV fits a logistic-regression probe on labeled brain examples. Case 2's shared embedding space allows something Case 1 cannot do: derive a concept direction directly from the *difference between two text-prototype embeddings*, then pull it back into the brain encoder's hidden space via the linear projection's transpose — no labeled brain examples needed at all. See `src/neurolens/concepts_case2.py`.

In [6]:
concept_results = run_case2_concept_analysis(model, test_loader, device, CLASS_NAMES)
for concept, res in concept_results.items():
    scores = {k: round(v, 2) for k, v in res["scores"].items()}
    print(f"{concept:12s}: {scores}")


hand        : {'baseline': 0.18, 'left_hand': 0.99, 'right_hand': 1.0, 'left_foot': 0.15, 'right_foot': 0.11, 'tongue': 0.2}
foot        : {'baseline': 0.58, 'left_hand': 0.18, 'right_hand': 0.28, 'left_foot': 0.99, 'right_foot': 0.99, 'tongue': 0.33}
tongue      : {'baseline': 0.35, 'left_hand': 0.07, 'right_hand': 0.13, 'left_foot': 0.23, 'right_foot': 0.26, 'tongue': 1.0}
right_side  : {'baseline': 0.38, 'left_hand': 0.44, 'right_hand': 0.26, 'left_foot': 0.56, 'right_foot': 0.47, 'tongue': 0.56}
left_side   : {'baseline': 0.62, 'left_hand': 0.56, 'right_hand': 0.74, 'left_foot': 0.44, 'right_foot': 0.53, 'tongue': 0.44}


**Reading it**: `hand`, `foot`, and `tongue` separate almost perfectly (TCAV 0.98–1.0 for the matching class) — evidence the representation organizes movement information along human-interpretable axes. `left_side`/`right_side` are markedly weaker and noisier (0.26–0.74). This is the same asymmetry Case 1's labeled-example CAV found independently — two unrelated derivation methods agreeing is stronger evidence it's a real property of the learned representation, not a probe artifact of either method.

## Part 3 — Text-conditioned attribution (a capability unique to Case 2)

Case 1's RSN attribution explains one of 6 fixed classifier logits. Case 2 has no classifier, but its brain-text similarity score plays exactly the same role a logit does — and because the shared embedding space accepts *any* text, not just the 6 known conditions, attribution can be computed for an arbitrary literature-derived sentence. Concretely: embed the sentence with the same frozen MiniLM + trained projection Case 2 already uses, then ask which resting-state network makes this specific window's brain embedding align with that specific sentence.

In [7]:
# find a real left_hand window
x0 = None
for batch in test_loader:
    for i in range(batch["x"].shape[0]):
        if int(batch["y"][i].item()) == 1:
            x0 = batch["x"][i : i + 1]
            break
    if x0 is not None:
        break

for phrase in ["left hand movement is contralateral", "tongue representation is bilateral"]:
    result = compare_methods_text(model, embedding_model_st, phrase, x0, network_indices, device)
    print(f'--- query: "{phrase}" ---')
    for method in ["saliency", "integrated_gradients", "shapley", "lime"]:
        top = NETWORK_NAMES[int(abs(result[method]["normalized"]).argmax())]
        vals = {n: round(float(v), 3) for n, v in zip(NETWORK_NAMES, result[method]["normalized"])}
        print(f"  {method:20s} top={top:6s} | {vals}")


--- query: "left hand movement is contralateral" ---
  saliency             top=SomMot | {'Vis': 0.17, 'SomMot': 0.254, 'DorsAttn': 0.106, 'SalVentAttn': 0.106, 'Limbic': 0.059, 'Cont': 0.111, 'Default': 0.194}
  integrated_gradients top=SomMot | {'Vis': 0.161, 'SomMot': 0.295, 'DorsAttn': 0.098, 'SalVentAttn': 0.093, 'Limbic': 0.054, 'Cont': 0.102, 'Default': 0.197}
  shapley              top=SomMot | {'Vis': 0.123, 'SomMot': 0.537, 'DorsAttn': 0.124, 'SalVentAttn': 0.143, 'Limbic': 0.004, 'Cont': 0.053, 'Default': 0.015}
  lime                 top=SomMot | {'Vis': 0.108, 'SomMot': 0.542, 'DorsAttn': 0.156, 'SalVentAttn': 0.134, 'Limbic': 0.005, 'Cont': 0.046, 'Default': 0.009}


--- query: "tongue representation is bilateral" ---
  saliency             top=SomMot | {'Vis': 0.174, 'SomMot': 0.261, 'DorsAttn': 0.098, 'SalVentAttn': 0.102, 'Limbic': 0.067, 'Cont': 0.107, 'Default': 0.191}
  integrated_gradients top=SomMot | {'Vis': 0.153, 'SomMot': 0.289, 'DorsAttn': 0.084, 'SalVentAttn': 0.095, 'Limbic': 0.076, 'Cont': 0.098, 'Default': 0.206}
  shapley              top=SomMot | {'Vis': 0.117, 'SomMot': 0.655, 'DorsAttn': 0.013, 'SalVentAttn': 0.082, 'Limbic': 0.106, 'Cont': 0.022, 'Default': 0.005}
  lime                 top=SomMot | {'Vis': 0.106, 'SomMot': 0.661, 'DorsAttn': 0.008, 'SalVentAttn': 0.077, 'Limbic': 0.101, 'Cont': 0.026, 'Default': 0.021}


**Reading it**: all four methods agree SomMot dominates for both queries on this real left_hand window — the neuroanatomically expected answer for a motor query, and consistent whether the query text is about hand movement or (for contrast) tongue movement. That the attribution pattern doesn't sharply discriminate between two different text queries *on the same window* is itself an honest, informative observation: for a window whose real content is dominated by one network's activity, that network tends to carry most of the similarity signal to almost any motor-related text, not just the "correct" one — a limit worth remembering before treating this as a fine-grained discriminative tool between similar candidate concepts. It remains a genuine complement to CAV/TCAV: attribution is local (per-window, per-network), CAV is global (aggregated direction, many windows).

## Part 4 — The CAV-RAG loop, with a deterministic-verdict fix

**The bug, as measured**: asked to freely judge whether the CAV evidence agrees with the literature, the LLM defaulted to AGREE regardless of the actual TCAV score (measured previously at 10/12 real cases — see `docs/project-summary.md` §3.6). A LoRA fine-tune aimed at teaching the correct judgment (see `16_rag_llm_improvements.ipynb`) fixed format compliance but not the underlying reasoning at the data scale tried.

**The fix**: stop asking the LLM to decide the verdict at all. Its own stance label (SUPPORTS/CONTRADICTS/UNRELATED, extracted per excerpt — now in the same call as the concept phrase, at no extra cost) plus the measured TCAV score are enough to compute `AGREE`/`DISAGREE`/`UNCLEAR` deterministically in code (`expected_verdict_from_stance_and_tcav`). The LLM's remaining job is to narrate a conclusion it's given, not reach one — so "faithfulness" is now a narrower, more tractable question: did the write-up actually mention every verdict it was told, rather than silently dropping or inverting one (`score_synthesis_reporting_accuracy`).

In [8]:
chunks, embeddings = load_index(PROJECT_ROOT / "artifacts" / "paper_index")
reranker = load_reranker()
generate_fn = make_mlx_generate_fn()
print("ready. corpus chunks:", len(chunks))


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7279.01it/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Fetching 6 files: 100%|██████████| 6/6 [00:00<00:00, 4110.05it/s]

ready. corpus chunks: 879


In [9]:
import random

random.seed(0)
by_class: dict[int, list] = {c: [] for c in range(6)}
for batch in test_loader:
    for i in range(batch["x"].shape[0]):
        y = int(batch["y"][i].item())
        if len(by_class[y]) < 2:
            by_class[y].append((batch, i))
    if all(len(v) >= 2 for v in by_class.values()):
        break

sampled = []
for c in range(6):
    sampled.extend(by_class[c][:1])  # one per class = 6 examples

print(f"running the fixed CAV-RAG loop on {len(sampled)} examples ...")
all_results = []
for n, (batch, i) in enumerate(sampled):
    x0 = batch["x"][i : i + 1]
    t0 = time.time()
    result = explain_decoded_window_with_cav_loop_case2(
        contrastive_model=model, x=x0,
        subject_id=batch["subject_id"][i], task=batch["task"][i], run=batch["run"][i],
        class_to_condition=CLASS_TO_CONDITION, device=device,
        embedding_model=embedding_model_st, corpus_chunks=chunks, corpus_embeddings=embeddings,
        cav_test_loader=test_loader, generate_fn=generate_fn, reranker=reranker, candidate_k=20, top_k=5,
    )
    elapsed = time.time() - t0
    all_results.append(result)
    print("=" * 100)
    print(f"[{n+1}/{len(sampled)}] subject={result['subject_id']} decoded={result['condition']} "
          f"confidence={result['decoded']['confidence']:.2f} ({elapsed:.0f}s)")
    for p in result["extracted_concept_phrases"]:
        print(f"  phrase: \"{p['phrase']}\" (stance={p.get('stance')}, from {p['source_file']})")
    for r in result["cav_loop_results"]:
        if not r.get("matched_concepts"):
            continue
        for concept, vals in r["results"].items():
            print(f"    -> {concept}: TCAV={vals['tcav_score_for_decoded_class']:.2f}  COMPUTED VERDICT={vals['verdict']}")
    print("reporting accuracy:", result["reporting_accuracy"])
    print("SYNTHESIS:", result["final_synthesis"][:500])


running the fixed CAV-RAG loop on 6 examples ...


[1/6] subject=102311 decoded=baseline confidence=0.97 (29s)
  phrase: "Motor mapping task activates left hand area." (stance=SUPPORTS, from 1-s2.0-S1053811913005272-main.pdf)
  phrase: "Motor task involves hand and foot movements." (stance=SUPPORTS, from 1-s2.0-S1053811913005272-main.pdf)
  phrase: "motor areas localized using cytoarchitectural population maps." (stance=SUPPORTS, from ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf)
  phrase: "Activation in left motor cortex confirmed." (stance=SUPPORTS, from 1-s2.0-S1053811913005351-main.pdf)
  phrase: "Motor tasks elicit strong activation in motor cortex." (stance=SUPPORTS, from 1-s2.0-S1053811913005272-main.pdf)
    -> hand: TCAV=0.18  COMPUTED VERDICT=DISAGREE
    -> hand: TCAV=0.18  COMPUTED VERDICT=DISAGREE
    -> foot: TCAV=0.58  COMPUTED VERDICT=UNCLEAR
reporting accuracy: {'n_given_verdicts': 3, 'given_verdicts': [{'concept': 'hand', 'tcav_score': 0.182

[2/6] subject=102311 decoded=left_hand confidence=0.97 (30s)
  phrase: "Left hand is neurally represented in the decoded result." (stance=SUPPORTS, from 1-s2.0-S1053811913005272-main.pdf)
  phrase: "Left hand is neurally represented ventrally." (stance=SUPPORTS, from 1-s2.0-S1053811913005272-main.pdf)
  phrase: "Left lateral parietal cortex involved in motor tasks." (stance=SUPPORTS, from 1-s2.0-S1053811913005272-main.pdf)
  phrase: "Left hand movement type is neurally represented." (stance=SUPPORTS, from bhx179.pdf)
  phrase: "motor task, specifically hand, is neurally represented." (stance=SUPPORTS, from ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf)
    -> hand: TCAV=0.99  COMPUTED VERDICT=AGREE
    -> right_side: TCAV=0.44  COMPUTED VERDICT=UNCLEAR
    -> left_side: TCAV=0.56  COMPUTED VERDICT=UNCLEAR
reporting accuracy: {'n_given_verdicts': 3, 'given_verdicts': [{'concept': 'hand', 'tcav_score': 0.9850427

[3/6] subject=102311 decoded=right_hand confidence=1.00 (33s)
  phrase: "Right hand is neurally represented." (stance=SUPPORTS, from 1-s2.0-S1053811913005272-main.pdf)
  phrase: "Right hand represented in human motor cortex." (stance=SUPPORTS, from meier-et-al-2008-complex-organization-of-human-primary-motor-cortex-a-high-resolution-fmri-study.pdf)
  phrase: "Right hand motor task is neurally represented." (stance=SUPPORTS, from bhx179.pdf)
  phrase: "Right hand is neurally represented via brain-text joint-embedding." (stance=SUPPORTS, from ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf)
  phrase: "Right hand motor imagery activates multiple brain areas." (stance=SUPPORTS, from ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf)
    -> hand: TCAV=1.00  COMPUTED VERDICT=AGREE
reporting accuracy: {'n_given_verdicts': 1, 'given_verdicts': [{'c

[4/6] subject=102311 decoded=left_foot confidence=0.99 (37s)
  phrase: "Left foot is neurally represented on the left side." (stance=SUPPORTS, from 1-s2.0-S1053811913005272-main.pdf)
  phrase: "Left foot activation confirmed via brain imaging." (stance=SUPPORTS, from 1-s2.0-S1053811913005272-main.pdf)
  phrase: "Left foot movement type neurally represented." (stance=SUPPORTS, from bhx179.pdf)
  phrase: "Left foot motor activity detected in brain." (stance=SUPPORTS, from ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf)
  phrase: "Left foot is neurally represented via brain-text joint-embedding." (stance=SUPPORTS, from ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf)
    -> foot: TCAV=0.99  COMPUTED VERDICT=AGREE
reporting accuracy: {'n_given_verdicts': 1, 'given_verdicts': [{'concept': 'foot', 'tcav_score': 0.9880952380952381, 'verdict': '

[5/6] subject=102311 decoded=right_foot confidence=0.98 (33s)
  phrase: "Right foot activation confirmed with high confidence level." (stance=SUPPORTS, from 1-s2.0-S1053811913005272-main.pdf)
  phrase: "Right foot activation confirmed in frontal cortex." (stance=SUPPORTS, from 1-s2.0-S1053811913005272-main.pdf)
  phrase: "right foot movement type is neurally represented." (stance=SUPPORTS, from bhx179.pdf)
  phrase: "Right foot is neurally represented in motor areas." (stance=SUPPORTS, from ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf)
  phrase: "Right foot represented in human motor cortex." (stance=SUPPORTS, from meier-et-al-2008-complex-organization-of-human-primary-motor-cortex-a-high-resolution-fmri-study.pdf)
    -> foot: TCAV=0.99  COMPUTED VERDICT=AGREE
reporting accuracy: {'n_given_verdicts': 1, 'given_verdicts': [{'concept': 'foot', 'tcav_score': 0.9920634920634921, 'verdict': 'AGREE'}], 'n_mentione

[6/6] subject=102311 decoded=tongue confidence=0.94 (34s)
  phrase: "Tongue activation observed in motor mapping task." (stance=SUPPORTS, from 1-s2.0-S1053811913005272-main.pdf)
  phrase: "Tongue activation found in ventral lateral prefrontal cortex." (stance=SUPPORTS, from 1-s2.0-S1053811913005272-main.pdf)
  phrase: "Tongue representation is neurally localized to central sulcus." (stance=SUPPORTS, from ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf)
  phrase: "Tongue motion represented in left hemisphere's 400-area cerebral cortex." (stance=SUPPORTS, from bhx179.pdf)
  phrase: "Imagery of tongue activates corresponding motor representations." (stance=SUPPORTS, from ehrsson-et-al-2003-imagery-of-voluntary-movement-of-fingers-toes-and-tongue-activates-corresponding-body-part-specific.pdf)
    -> tongue: TCAV=1.00  COMPUTED VERDICT=AGREE
    -> tongue: TCAV=1.00  COMPUTED VERDICT=AGREE
    -> right_side: TCAV=0.

In [10]:
out_path = RESULTS_DIR / "case2_rag_cav_loop_examples_fixed.json"
with open(out_path, "w") as f:
    json.dump(all_results, f, indent=2)
print("saved", out_path)

n_with_verdicts = sum(1 for r in all_results if r["reporting_accuracy"]["n_given_verdicts"] > 0)
n_fully_mentioned = sum(1 for r in all_results if r["reporting_accuracy"]["all_mentioned"])
print(f"\n{n_with_verdicts}/{len(all_results)} examples had at least one computed verdict to report;"
      f" {n_fully_mentioned}/{n_with_verdicts} of those mentioned ALL given verdicts in the write-up.")
print("\nCritically: every computed verdict above is correct BY CONSTRUCTION - it never depends on the LLM's judgment,"
      " so the original failure mode (defaulting to AGREE regardless of the TCAV score) can no longer occur in the"
      " reported verdict, only (at most) in how completely the prose narrates it.")


saved /Users/srinivasgovindasurampudi/Projects/neurolens-rag/results/case2_rag_cav_loop_examples_fixed.json

6/6 examples had at least one computed verdict to report; 5/6 of those mentioned ALL given verdicts in the write-up.

Critically: every computed verdict above is correct BY CONSTRUCTION - it never depends on the LLM's judgment, so the original failure mode (defaulting to AGREE regardless of the TCAV score) can no longer occur in the reported verdict, only (at most) in how completely the prose narrates it.


## Summary

- **Modality mixing**: confirmed genuine, not an artifact of retrieval ranking (gap ratio ≈1.0, brain-only silhouette 0.43, PCA visualization). R-precision 0.83–0.89 is the honest retrieval-quality summary (not the misleading precision-only 1.00 an earlier version of this project reported).
- **Case2-native CAV**: hand/foot/tongue near-perfectly separated (0.98–1.0), laterality weaker (0.26–0.74) — reproducing Case 1's finding via a completely independent, brain-example-free method.
- **Text-conditioned attribution**: a genuinely new capability (arbitrary free text, not just 6 known conditions) — complementary to CAV, though this single example shows it doesn't sharply discriminate between similar candidate texts on one window, worth more testing before relying on it for fine discrimination.
- **The faithfulness bug**: fixed at the root cause (verdict computed in code from stance+TCAV, not decided by the LLM) rather than trained around (the LoRA attempt in `16_rag_llm_improvements.ipynb` only partially worked). The reported agreement/disagreement conclusion is now correct by construction.

Findings folded into `docs/project-summary.md` §3.6.
